In [ ]:
from urllib.robotparser import RobotFileParser

def check_robots(url):
    """Check if scraping is allowed"""
    rp = RobotFileParser()
    rp.set_url('http://mtc-m16c.sid.inpe.br/robots.txt')
    rp.read()
    return rp.can_fetch('*', url)

if check_robots('http://mtc-m16c.sid.inpe.br/rep/8JMKD3MGPDW34P/42T25JL'):
    print("✅ Scraping allowed")
else:
    print("❌ Scraping blocked by robots.txt")

✅ Scraping allowed


In [ ]:
import json
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

URL = "http://mtc-m16c.sid.inpe.br/rep/8JMKD3MGPDW34P/42T25JL"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(URL, headers=headers, timeout=30)
response.raise_for_status()
response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")

body_frame = soup.find("frame", {"name": "body"})
body_src = body_frame.get("src")
url_body = urljoin(URL, body_src)

print("URL do frame body:")
print(url_body)

response_body = requests.get(url_body, headers=headers, timeout=30)
response_body.raise_for_status()
response_body.encoding = "utf-8"

soup_body = BeautifulSoup(response_body.text, "html.parser")

links_artigos = []

for tabela in soup_body.find_all("table", class_="titleAuthorTABLE"):
    a = tabela.find("a", href=True)

    if a and a["href"].startswith("goto/"):
        href = a["href"]

        repo = href.split("?")[0].replace("goto/", "")

        link_artigo = (
            "http://mtc-m16c.sid.inpe.br/col/"
            + repo
            + "/doc/thisInformationItemHomePage.html"
        )

        links_artigos.append(link_artigo)

links_artigos = list(dict.fromkeys(links_artigos))

print("Links de artigos encontrados:", len(links_artigos))
links_artigos

print("Links de artigos encontrados:", len(links_artigos))
links_artigos[:5]

def abrir_pagina(url, tentativas=3):
    for tentativa in range(tentativas):
        try:
            response = requests.get(url, headers=headers, timeout=60)
            response.raise_for_status()
            response.encoding = "utf-8"
            return BeautifulSoup(response.text, "html.parser")

        except requests.exceptions.RequestException as e:
            print(f"Tentativa {tentativa + 1} falhou em: {url}")
            print(e)
            time.sleep(5)

    raise Exception(f"Não consegui abrir a página depois de {tentativas} tentativas: {url}")

print("Tabelas de artigos:", len(soup_body.find_all("table", class_="titleAuthorTABLE")))

for tabela in soup_body.find_all("table", class_="titleAuthorTABLE"):
    a = tabela.find("a", href=True)
    print(a["href"] if a else "sem link")

URL do frame body:
http://mtc-m16c.sid.inpe.br/displaydoccontent.cgi/sid.inpe.br/mtc-m18/2014/10.14.15.55?nonemptyquerystring=couldBeAnyValue
Links de artigos encontrados: 35
Links de artigos encontrados: 35
Tabelas de artigos: 35
goto/dpi.inpe.br/geoinfo@80/2006/08.21.14.55?ibiurl.backgroundlanguage=pt-BR&searchinputvalue=&parentidentifiercitedby=8JMKD3MGP8W/3H8DCF8&forcerecentflag=0&forcehistorybackflag=1
goto/dpi.inpe.br/geoinfo@80/2006/08.21.12.29?ibiurl.backgroundlanguage=pt-BR&searchinputvalue=&parentidentifiercitedby=8JMKD3MGP8W/3H8DCF8&forcerecentflag=0&forcehistorybackflag=1
goto/dpi.inpe.br/geoinfo@80/2006/08.21.14.46?ibiurl.backgroundlanguage=pt-BR&searchinputvalue=&parentidentifiercitedby=8JMKD3MGP8W/3H8DCF8&forcerecentflag=0&forcehistorybackflag=1
goto/dpi.inpe.br/geoinfo@80/2006/08.21.13.28?ibiurl.backgroundlanguage=pt-BR&searchinputvalue=&parentidentifiercitedby=8JMKD3MGP8W/3H8DCF8&forcerecentflag=0&forcehistorybackflag=1
goto/dpi.inpe.br/geoinfo@80/2006/08.18.18.57?ibiu

In [96]:
from urllib.parse import urljoin

def encontrar_link_metadata(url_artigo):
    soup_artigo = abrir_pagina(url_artigo)

    header = soup_artigo.find("frame", {"name": "header"})

    if header is None:
        return None

    url_header = header.get("src")

    soup_header = abrir_pagina(url_header)

    for a in soup_header.find_all("a", href=True):
        texto = a.get_text(" ", strip=True).lower()

        if "metadados" in texto or "metadata" in texto:
            return a["href"]

    return None


links_metadata = []

for link in links_artigos:
    try:
        metadata_url = encontrar_link_metadata(link)

        if metadata_url:
            links_metadata.append(metadata_url)
            print("Metadata:", metadata_url)
        else:
            print("Sem metadata:", link)

        time.sleep(1)

    except Exception as e:
        print("Erro em:", link)
        print(e)

links_metadata = list(dict.fromkeys(links_metadata))

print("Total de metadados encontrados:", len(links_metadata))

Metadata: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.21.14.55.32?ibiurl.backgroundlanguage=pt-BR
Metadata: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.21.12.29.45?ibiurl.backgroundlanguage=pt-BR
Metadata: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.21.14.46.26?ibiurl.backgroundlanguage=pt-BR
Metadata: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.21.13.28.34?ibiurl.backgroundlanguage=pt-BR
Metadata: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.18.18.57.51?ibiurl.backgroundlanguage=en
Metadata: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.21.12.21.05?ibiurl.backgroundlanguage=pt-BR
Metadata: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.21.13.44.07?ibiurl.backgroundlanguage=en
Metadata: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.21.14.21.12?ibiurl.backgroundlanguage=pt-BR
Metadata: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.18.18.35.32?ibiurl.backgroundlang

In [ ]:
from pathlib import Path
import pandas as pd
import re
import time

ANO = 2012
N_EDICAO = 13
PASTA = Path(str(ANO))
PASTA.mkdir(exist_ok=True)

def limpar_texto(x):
    if not x:
        return None
    return " ".join(x.replace("�", "ã").split())

def get_field(soup, codigo):
    for abbr in soup.find_all("abbr"):
        title = abbr.get("title", "")

        if codigo in title.split():
            td_chave = abbr.find_parent("td")
            if td_chave is None:
                continue

            td_valor = td_chave.find_next_sibling("td")
            if td_valor is None:
                continue

            return td_valor.get_text("\n", strip=True)

    return None

def extrair_lista_numerada(texto):
    if not texto:
        return []

    linhas = texto.split("\n")
    resultado = []

    for linha in linhas:
        linha = re.sub(r"^\d+\s+", "", linha).strip()
        if linha:
            resultado.append(limpar_texto(linha))

    return resultado

def formatar_nome(nome):
    if "," in nome:
        sobrenome, nomes = nome.split(",", 1)
        return f"{nomes.strip()} {sobrenome.strip()}"
    return nome

def calcular_n_paginas(paginas):
    if not paginas:
        return pd.NA

    m = re.search(r"(\d+)\s*-\s*(\d+)", paginas)
    if m:
        return int(m.group(2)) - int(m.group(1)) + 1

    return pd.NA

def normalizar_tipo(tipo):
    if not tipo:
        return None

    tipo = tipo.lower()

    if "full" in tipo:
        return "full"
    if "short" in tipo:
        return "short"

    return tipo

def normalizar_lingua(lingua):
    if not lingua:
        return None

    lingua = lingua.lower()

    if lingua == "en":
        return "english"
    if lingua == "pt":
        return "portuguese"

    return lingua

def extrair_artigo(url_metadata):
    soup = abrir_pagina(url_metadata)

    titulo = get_field(soup, "tit")
    abstract = get_field(soup, "ab")
    tipo = get_field(soup, "tertiaryty")
    lingua = get_field(soup, "lan")
    autores_raw = get_field(soup, "au")
    afiliacoes_raw = get_field(soup, "af")
    paginas_raw = get_field(soup, "pag")
    local_evento = get_field(soup, "conferencel")

    autores = [formatar_nome(a) for a in extrair_lista_numerada(autores_raw)]
    instituicoes = extrair_lista_numerada(afiliacoes_raw)

    return {
        "titulo": limpar_texto(titulo),
        "abstract": limpar_texto(abstract),
        "n_paginas": calcular_n_paginas(paginas_raw),
        "tipo_artigo": normalizar_tipo(tipo),
        "lingua": normalizar_lingua(lingua),
        "autores": autores,
        "instituicoes": instituicoes,
        "local_evento": limpar_texto(local_evento),
        "url_metadata": url_metadata
    }

In [98]:
dados_artigos = []
dados_autoria = []
dados_pessoas = []

mapa_pessoas = {}
id_artigo = 1
id_autoria = 1
id_pessoa = 1
local_evento_final = pd.NA

for url in links_metadata:
    print("Extraindo:", url)

    try:
        artigo = extrair_artigo(url)
    except Exception as e:
        print("Pulando por erro:", url)
        print(e)
        continue

    if artigo["local_evento"]:
        local_evento_final = artigo["local_evento"]

    dados_artigos.append({
        "id_artigo": id_artigo,
        "titulo": artigo["titulo"],
        "abstract": artigo["abstract"],
        "n_paginas": artigo["n_paginas"],
        "tipo_artigo": artigo["tipo_artigo"],
        "lingua": artigo["lingua"],
        "n_edicao": N_EDICAO
    })

    for ordem, nome in enumerate(artigo["autores"], start=1):
        instituicao = (
            artigo["instituicoes"][ordem - 1]
            if ordem - 1 < len(artigo["instituicoes"])
            else pd.NA
        )

        chave = (nome, instituicao)

        if chave not in mapa_pessoas:
            mapa_pessoas[chave] = id_pessoa

            dados_pessoas.append({
                "id_pessoa": id_pessoa,
                "nome": nome,
                "instituicao": instituicao
            })

            id_pessoa += 1

        dados_autoria.append({
            "id_autoria": id_autoria,
            "id_artigo": id_artigo,
            "id_pessoa": mapa_pessoas[chave],
            "ordem_autoria": ordem,
            "autor_correspondente": False
        })

        id_autoria += 1

    id_artigo += 1
    time.sleep(1)

artigos = pd.DataFrame(dados_artigos)
autoria = pd.DataFrame(dados_autoria)
pessoas = pd.DataFrame(dados_pessoas)

evento = pd.DataFrame([{
    "id_evento": 1,
    "n_edicao": N_EDICAO,
    "ano": ANO,
    "local": local_evento_final
}])

comite_cientifico = pd.DataFrame(columns=[
    "id_revisor", "id_pessoa", "n_edicao"
])

organizacao = pd.DataFrame(columns=[
    "id_organizador", "id_pessoa", "funcao", "n_edicao"
])

artigos.to_csv(PASTA / "artigos.csv", index=False, encoding="utf-8")
autoria.to_csv(PASTA / "autoria.csv", index=False, encoding="utf-8")
pessoas.to_csv(PASTA / "pessoas.csv", index=False, encoding="utf-8")
evento.to_csv(PASTA / "evento.csv", index=False, encoding="utf-8")
comite_cientifico.to_csv(PASTA / "comite_cientifico.csv", index=False, encoding="utf-8")
organizacao.to_csv(PASTA / "organização.csv", index=False, encoding="utf-8")

artigos.head()

Extraindo: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.21.14.55.32?ibiurl.backgroundlanguage=pt-BR
Extraindo: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.21.12.29.45?ibiurl.backgroundlanguage=pt-BR
Extraindo: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.21.14.46.26?ibiurl.backgroundlanguage=pt-BR
Extraindo: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.21.13.28.34?ibiurl.backgroundlanguage=pt-BR
Extraindo: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.18.18.57.51?ibiurl.backgroundlanguage=en
Extraindo: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.21.12.21.05?ibiurl.backgroundlanguage=pt-BR
Extraindo: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.21.13.44.07?ibiurl.backgroundlanguage=en
Extraindo: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.21.14.21.12?ibiurl.backgroundlanguage=pt-BR
Extraindo: http://mtc-m16c.sid.inpe.br/dpi.inpe.br/geoinfo@80/2006/08.18.18.35.32?ibiurl.backg

,id_artigo,titulo,abstract,n_paginas,tipo_artigo,lingua,n_edicao
0,1,The multi-layered interval categorizer tessell...,The paper presents the results obtained by an ...,18,None,english,6
1,2,VOROMARKETING: um sistema parametrizãvel para ...,"Este artigo apresenta o VoroMarketing, um sist...",12,None,portuguese,6
2,3,Approximate spatial query processing using ras...,"Nowadays, the database characteristics, such a...",19,None,english,6
3,4,Geo-object catalogs to enable geographic datab...,This paper introduces the concept of Ontology ...,12,None,english,6
4,5,Temporal constraints between cyclic geographic...,This paper presents a data model for cyclic ge...,17,None,english,6


In [ ]:
import io
import re
import unicodedata
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pypdf import PdfReader

URL_ANAIS = "http://mtc-m21b.sid.inpe.br/col/sid.inpe.br/mtc-m21b/2015/12.16.18.10/doc/thisInformationItemHomePage.html"

def normalizar_chave_nome(nome):
    nome = str(nome).strip().lower()
    nome = unicodedata.normalize("NFD", nome)
    nome = "".join(c for c in nome if unicodedata.category(c) != "Mn")
    nome = re.sub(r"\s+", " ", nome)
    return nome

def baixar_pdf_anais(url):
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()

    content_type = r.headers.get("content-type", "").lower()

    if "pdf" in content_type:
        return r.content

    soup = BeautifulSoup(r.text, "html.parser")

    frame_body = soup.find("frame", {"name": "body"})
    if frame_body:
        pdf_url = urljoin(url, frame_body.get("src"))
        r_pdf = requests.get(pdf_url, headers=headers, timeout=60)
        r_pdf.raise_for_status()
        return r_pdf.content

    link_pdf = soup.find("a", href=lambda h: h and ".pdf" in h.lower())
    if link_pdf:
        pdf_url = urljoin(url, link_pdf.get("href"))
        r_pdf = requests.get(pdf_url, headers=headers, timeout=60)
        r_pdf.raise_for_status()
        return r_pdf.content

    raise Exception("Não consegui encontrar o PDF dos anais.")

def extrair_texto_pdf(pdf_bytes):
    reader = PdfReader(io.BytesIO(pdf_bytes))
    textos = []

    for page in reader.pages:
        texto = page.extract_text()
        if texto:
            textos.append(texto)

    return "\n".join(textos)

def obter_id_pessoa(nome, instituicao):
    global pessoas

    chave = normalizar_chave_nome(nome)

    if "chave_nome" not in pessoas.columns:
        pessoas["chave_nome"] = pessoas["nome"].apply(normalizar_chave_nome)

    existentes = pessoas[pessoas["chave_nome"] == chave]

    if len(existentes) > 0:
        return int(existentes.iloc[0]["id_pessoa"])

    novo_id = int(pessoas["id_pessoa"].max()) + 1 if len(pessoas) > 0 else 1

    nova_pessoa = pd.DataFrame([{
        "id_pessoa": novo_id,
        "nome": nome,
        "instituicao": instituicao,
        "chave_nome": chave
    }])

    pessoas = pd.concat([pessoas, nova_pessoa], ignore_index=True)

    return novo_id

def limpar_linhas(texto):
    linhas = []

    for linha in texto.split("\n"):
        linha = linha.strip()
        linha = re.sub(r"\s+", " ", linha)

        if linha:
            linhas.append(linha)

    return linhas

In [100]:
pdf_bytes = baixar_pdf_anais(URL_ANAIS)
texto_anais = extrair_texto_pdf(pdf_bytes)
linhas = limpar_linhas(texto_anais)

# Só para conferir
for i, linha in enumerate(linhas[:80]):
    print(i, linha)

0 Proceedings
1 La´ ercio M. Namikawa and Vania Bogorny (Eds.)
2 Dados Internacionais de Catalogação na Publicação
3 SI57a Simpósio Brasileiro de Geoinformática (11. : 201 2: Campos do Jordão,SP)
4 Anais do 13º Simpósio Brasileiro de Geoinformática, Campos do
5 Jordão, SP, 25 a 27 de novembro de 2012. / editado por Laércio Massaru
6 Namikawa (INPE), Vania Bogorny (UFSC) – São José dos Campos,
7 SP: MCTI/INPE, 2012.
8 CD + On-line
9 ISSN 2179-4820
10 1. Geoinformação. 2.Bancos de dados espaciais. 3.Análise
11 Espacial. 4. Sistemas de Informação Geográfica ( SIG). 5 .Dados
12 espaço-temporais. I. Namikawa, L.M. II. Bogorny, V. III. Título.
13 CDU: 681.3.06
14 Preface
15 This volume of proceedings contains papers presented at the XIII Brazilian Symposium on Geoinformatics, GeoInfo
16 2012, held in Campos do Jord˜ ao, Brazil, November 25-27, 2012. The GeoInfo conference series, inaugurated in
17 1999, reached its thirteenth edition in 2012. GeoInfo continues to consolidate itself as the mo

In [101]:
# ==========================
# Organização
# ==========================

organizadores_extraidos = []

funcoes = {
    "General Chair": "General Chair",
    "Program Chair": "Program Chair",
    "Local Organization": "Local Organization"
}

funcao_atual = None
i = 0

while i < len(linhas):
    linha = linhas[i]

    if linha in funcoes:
        funcao_atual = funcoes[linha]
        i += 1
        continue

    if funcao_atual and linha in ["Support", "Program committee", "Program Committee"]:
        funcao_atual = None

    if funcao_atual:
        nome = linha

        if i + 1 < len(linhas):
            instituicao = linhas[i + 1]
        else:
            instituicao = pd.NA

        if nome not in funcoes and instituicao not in funcoes:
            organizadores_extraidos.append({
                "nome": nome,
                "instituicao": instituicao,
                "funcao": funcao_atual
            })

            i += 2
            continue

    i += 1

organizadores_extraidos

[{'nome': 'Laercio Massaru Namikawa',
  'instituicao': 'National Institute for Space Research, INPE',
  'funcao': 'General Chair'},
 {'nome': 'Vania Bogorny',
  'instituicao': 'Federal University of Santa Catarina, UFSC',
  'funcao': 'Program Chair'},
 {'nome': 'Daniela Seki',
  'instituicao': 'INPE',
  'funcao': 'Local Organization'},
 {'nome': 'Janete da Cunha',
  'instituicao': 'INPE',
  'funcao': 'Local Organization'},
 {'nome': 'Luciana Moreira',
  'instituicao': 'INPE',
  'funcao': 'Local Organization'}]

In [102]:
# ==========================
# Comitê científico
# ==========================
from pathlib import Path
import pandas as pd

pessoas = pd.read_csv(PASTA / "pessoas.csv")

inicio = None
fim = None

for i, linha in enumerate(linhas):
    if linha.lower() == "program committee":
        inicio = i + 1
        break

if inicio is not None:
    for j in range(inicio, len(linhas)):
        linha = linhas[j]

        if linha.lower() in [
            "table of contents",
            "contents",
            "preface",
            "foreword"
        ]:
            fim = j
            break

    if fim is None:
        fim = len(linhas)

    bloco = linhas[inicio:fim]

    for linha in bloco:
        if "," in linha:
            partes = linha.split(",", 1)
            nome = partes[0].strip()
            instituicao = partes[1].strip()

            if nome and instituicao:
                revisores_extraidos.append({
                    "nome": nome,
                    "instituicao": instituicao
                })

# ==========================
# Atualizar pessoas
# ==========================

dados_organizacao = []
dados_comite = []

revisores_extraidos = list({
    normalizar_chave_nome(rev["nome"]): rev
    for rev in revisores_extraidos
}.values())

id_organizador = 1
for org in organizadores_extraidos:
    id_pessoa = obter_id_pessoa(org["nome"], org["instituicao"])

    dados_organizacao.append({
        "id_organizador": id_organizador,
        "id_pessoa": id_pessoa,
        "funcao": org["funcao"],
        "n_edicao": N_EDICAO
    })

    id_organizador += 1

id_revisor = 1
for rev in revisores_extraidos:
    id_pessoa = obter_id_pessoa(rev["nome"], rev["instituicao"])

    dados_comite.append({
        "id_revisor": id_revisor,
        "id_pessoa": id_pessoa,
        "n_edicao": N_EDICAO
    })

    id_revisor += 1

organizacao = pd.DataFrame(dados_organizacao)
comite_cientifico = pd.DataFrame(dados_comite)

# Remover coluna auxiliar antes de salvar
pessoas_final = pessoas.drop(columns=["chave_nome"], errors="ignore")

pessoas_final.to_csv(PASTA / "pessoas.csv", index=False, encoding="utf-8")
organizacao.to_csv(PASTA / "organização.csv", index=False, encoding="utf-8")
comite_cientifico.to_csv(PASTA / "comite_cientifico.csv", index=False, encoding="utf-8")

print("Organizadores:", len(organizacao))
print("Revisores:", len(comite_cientifico))
print("Pessoas:", len(pessoas_final))

organizacao.head(), comite_cientifico.head(), pessoas_final.tail()

Organizadores: 5
Revisores: 55
Pessoas: 163


(   id_organizador  id_pessoa              funcao  n_edicao
 0               1        108       General Chair         6
 1               2        109       Program Chair         6
 2               3        110  Local Organization         6
 3               4        111  Local Organization         6
 4               5        112  Local Organization         6,
    id_revisor  id_pessoa  n_edicao
 0           1        113         6
 1           2        114         6
 2           3        115         6
 3           4        116         6
 4           5        117         6,
      id_pessoa                      nome             instituicao
 158        159            Silvana Amaral                    INPE
 159        160  Jo˜ ao Pedro C. Cordeiro                    INPE
 160        161              Sergio Rosim                    INPE
 161        162             Jussara Ortiz                    INPE
 162        163  Mario J. Gaspar da Silva  Universidade de Lisboa)